In [2]:
import os
import sys

# 1. Force the notebook to run from your project workspace root
project_root = r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression"
os.chdir(project_root)

# 2. Add the 'src' folder to Python's system path so it can see 'Linear_regression_01'
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print("Current Working Directory:", os.getcwd())
print("System path updated. Available modules:", os.listdir(src_path))

Current Working Directory: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression
System path updated. Available modules: ['Linear_regression_01', 'Linear_regression_01.egg-info']


In [3]:
import os
print(os.getcwd())


C:\Users\Greesha Vaishnavi\Desktop\dsprojects\linear_regression


In [4]:
import sys
print(sys.path)

['c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\python310.zip', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\DLLs', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire', '', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages', 'C:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\linear_regression\\src', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\win32', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\win32\\lib', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\lib\\site-packages\\pythonwin']


In [5]:
import os

print(os.path.exists("../src"))
print(os.path.exists("../src/Linear_regression_01"))

False
False


In [6]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(PROJECT_ROOT)
print(SRC_PATH)
print(sys.path[:3])

C:\Users\Greesha Vaishnavi\Desktop\dsprojects
C:\Users\Greesha Vaishnavi\Desktop\dsprojects\src
['C:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\src', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\python310.zip', 'c:\\Users\\Greesha Vaishnavi\\.conda\\envs\\lire\\DLLs']


In [7]:
import box
print(box.__version__)

7.4.1


In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir : Path
    STATUS_FILE : Path
    transformed_train_path : Path
    transformed_test_path : Path
    preprocessor_path : Path
    trained_model_file_path: Path
    


In [9]:
from Linear_regression_01.entity.config_entity import ModelTrainerConfig
from Linear_regression_01.utils.common import read_yaml, create_directories
from Linear_regression_01.constant import CONFIG_FILE_PATH
from Linear_regression_01.constant import MODEL_TRAINER_DIR_NAME
from Linear_regression_01.constant import MODEL_TRAINER_TRAINED_MODEL_DIR
from Linear_regression_01.constant import MODEL_TRAINER_TRAINED_MODEL_NAME
from Linear_regression_01.constant import PARAMS_FILE_PATH
from Linear_regression_01.constant import SCHEMA_FILE_PATH




In [10]:
# Configuration Manager


class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        config = self.config.model_trainer

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:

        config = self.config.model_trainer

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            transformed_train_path=config.transformed_train_path,
            transformed_test_path=config.transformed_test_path,
            preprocessor_path=config.preprocessor_path,
            trained_model_file_path=config.trained_model_file_path
        )

        return model_trainer_config

In [11]:
import os
import joblib
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from Linear_regression_01.entity.config_entity import ModelTrainerConfig
from Linear_regression_01.logging import logger

In [12]:

# Components

class ModelTrainer:

    def __init__(self, config: ModelTrainerConfig):
        self.config = config


    def train(self):

        logger.info("Loading transformed training and testing data")

        train_arr = np.load(self.config.transformed_train_path)  # data from transformation in np array format
        test_arr = np.load(self.config.transformed_test_path)

        # Split Features and Target

        X_train = train_arr[:, :-1]   #Take all rows Take all columns Except last
        y_train = train_arr[:, -1]    #Take all rows and only last column i.e, price

        X_test = test_arr[:, :-1]    # take all rows and all columns except last column i.e, price
        y_test = test_arr[:, -1]     # takes all rows and only last colum target varaliable price

        logger.info("Training Linear Regression Model")

        # Train Model

        model = LinearRegression()   #selecting model 

        model.fit(X_train, y_train)  # fitting best model

        logger.info("Model Training Completed")

        # Prediction

        train_prediction = model.predict(X_train)   
        test_prediction = model.predict(X_test)

        # Evaluation

        train_score = r2_score(y_train, train_prediction)  # checcking accuracy 
        test_score = r2_score(y_test, test_prediction)

        logger.info(f"Training R2 Score : {train_score}")
        logger.info(f"Testing R2 Score : {test_score}")

        # Save Model

        os.makedirs(
            os.path.dirname(self.config.trained_model_file_path),
            exist_ok=True
        )

        joblib.dump(
            model,                                 # saving model in model.pkl
            self.config.trained_model_file_path
        )

        logger.info("Model Saved Successfully")

        return model

In [13]:
# Pipeline

STAGE_NAME = "Model Trainer Stage"


class ModelTrainerTrainingPipeline:

    def __init__(self):
        pass

    def main(self):
        config = ConfigurationManager()
        model_config = config.config.model_trainer

        model_trainer_config = (                   # configuration manager
            config.get_model_trainer_config()
        )

        model_trainer = ModelTrainer(              # components
            config=model_trainer_config
        )

        model_trainer.train()

In [14]:
obj = ModelTrainerTrainingPipeline()

obj.main()

[2026-07-26 23:51:18,137: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-26 23:51:18,137: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-26 23:51:18,151: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-07-26 23:51:18,153: INFO: common: created directory at artifacts]
[2026-07-26 23:51:18,154: INFO: common: created directory at artifacts/model_trainer]
[2026-07-26 23:51:18,156: INFO: 3957079982: Loading transformed training and testing data]
[2026-07-26 23:51:18,158: INFO: 3957079982: Training Linear Regression Model]
[2026-07-26 23:51:18,162: INFO: 3957079982: Model Training Completed]
[2026-07-26 23:51:18,166: INFO: 3957079982: Training R2 Score : 0.6859438988560158]
[2026-07-26 23:51:18,168: INFO: 3957079982: Testing R2 Score : 0.6529242642153179]
[2026-07-26 23:51:18,176: INFO: 3957079982: Model Saved Successfully]
